# 演習1 解答編 ―― スレッドとパイプライン

> まず `ex01_threads.ipynb` を自分で解いてから読んでください。
> このノートには、答えを**自分で確かめるためのコード**も入っています。上から順に ▶ を押してください。

## 1-1 予測クイズの答え

- **1人で順番に** ⇒ 約 **600 ms**（100ms × 3個 × 2人分）
- **2つのスレッドで** ⇒ 約 **300 ms**（A と B が同時に進むので、片方分の時間で済む）

図で見ると、違いは「**重なっているかどうか**」だけです。

```
【1人で順番に】600 ms
  A  111112222233333................
  B  ...............111112222233333.
     └── A が働く ──┘└── B が働く ──┘     どちらか一方しか動いていない

【2つのスレッドで】300 ms
  A  111112222233333.
  B  111112222233333.
     └─ 2人が同時に働く ─┘                 B の '.' が消えた
```

スレッドを使っても、**1つの仕事が速くなるわけではありません**（A の仕事はどちらも300ms）。
速くなるのは「**待っている時間がなくなる**」からです。

用語で言えば：**レイテンシは変わらず、スループットが2倍**になりました。
スレッドが改善するのは常にスループット側です。ここを取り違えないでください。

（このプログラムは実行中に何も表示せず、各仕事の開始・終了時刻だけを記録して、
最後に `draw()` がまとめて図を描く作りです。複数のスレッドが同時に画面へ書き込むと
表示が乱れることがあるためで、その話は演習2で扱います。）

## 1-2 予測クイズの答え

- **待つ仕事** ⇒ 1200 → 600 → 300 ms。**人数に比例して**速くなります。
  待つのに計算回路は要らないので、コア数に関係なく何人でも同時に待てるからです
- **計算する仕事** ⇒ どこかで**頭打ち**になります。同時に計算できるのは
  そのマシンの計算回路の数までだからです。頭打ちの場所はマシンによって違い、
  Colab では 1→2 スレッドでもほとんど縮まないことがあります

「スレッドを増やせば増やすほど速くなる」が成り立つのは**待つ仕事だけ**です。
計算する仕事の上限はマシンごとに違うので、`hardware_concurrency()` を目安にしつつ、
最後は測って確かめます。「とりあえず多めに立てておく」は効かないどころか、
スレッドを切り替える手間の分だけ損をすることがあります。

## 発展課題1・2 の解答 ―― ボトルネックは動く

次のセルを実行すると、3つの場合の逐次／パイプラインの所要時間とボトルネックが表になって出ます。

In [ ]:
%%writefile ans01a.cpp
#include <iostream>
#include <iomanip>
#include <algorithm>
using namespace std;
const char* SN[3] = {"Read", "Infer", "Show"};

// n フレーム処理し終わるまでの時間(ms)を返す
int finish(int t[3], int n, bool pipeline) {
    int fin[3] = {0, 0, 0}, last = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < 3; j++) {
            int wait = pipeline ? fin[j] : last;
            int s = max(j ? fin[j - 1] : 0, wait);
            fin[j] = s + t[j];
            if (j == 2) last = fin[2];
        }
    return last;
}

void show(const char* label, int t[3], int n) {
    int a = finish(t, n, false), b = finish(t, n, true);
    int bn = 0;
    for (int j = 1; j < 3; j++) if (t[j] > t[bn]) bn = j;
    cout << left << setw(24) << label << right << fixed << setprecision(1)
         << setw(6) << a << "ms" << setw(6) << 1000.0 * n / a << "FPS"
         << setw(8) << b << "ms" << setw(6) << 1000.0 * n / b << "FPS"
         << "    " << SN[bn] << "(" << t[bn] << "ms)"
         << " 上限" << setprecision(0) << 1000.0 / t[bn] << "FPS\n";
}

int main() {
    int a[3] = {30, 30, 30};
    int b[3] = {13, 67, 33};      // 真ん中が重いとき
    int c[3] = {13,  2, 33};      // 真ん中だけを大幅に速くしたとき
    cout << "Read/Infer/Show          逐次            パイプライン       ボトルネック\n";
    cout << "-------------------------------------------------------------------------\n";
    show("30 / 30 / 30", a, 6);
    show("13 / 67 / 33", b, 6);
    show("13 /  2 / 33", c, 6);
    return 0;
}

In [ ]:
!g++ -std=c++17 ans01a.cpp -o ans01a && ./ans01a

**1. `{13, 67, 33}` ―― 真ん中が重い**

逐次 678ms（8.8 FPS）→ パイプライン 448ms（13.4 FPS）。1.5倍です。
図にすると Infer の行だけがびっしり埋まり、**Read と Show の行は `.` だらけ**になります。
十分に長く流し続けたときの上限は `1000 / 67 ≒ 15 FPS`。ここが天井です。

**2. `{13, 2, 33}` ―― 真ん中だけを大幅に速くした**

**ボトルネックは真ん中から Show へ移ります。**
上限は Show の 33ms で決まり、`1000 / 33 ≒ 30 FPS`。
真ん中を 67ms → 2ms と **33倍速く**したのに、全体は 15 FPS → 30 FPS の **2倍**にしかなりません。

> **一番遅い段を速くしない限り、全体は速くならない。**

これは **アムダールの法則** と呼ばれる考え方の、いちばん分かりやすい形です。
「速くした部分の効果は、それが全体に占める割合までしか出ない」。

そして、この状況で **真ん中の担当を増やすことにはまったく意味がありません**（発展課題3の答え）。
すでに手待ちの係を増やしても、詰まっている Show は1ミリも速くなりません。

## 発展課題3・4 の解答 ―― 人を増やしても倍にならないとき

発展課題3の答えは上で見たとおり「意味がない」です。ボトルネックが別の段にあるからです。

では発展課題4、**ボトルネックの段そのものを2人に増やしたら、必ず倍になる**のでしょうか。
なりません。代表的な理由が2つあります。

### 理由① コアが足りない

1-2 で見たとおりです。計算が中心の仕事（CPUバウンド）は、
「同時に計算できる数」で頭打ちになります。

### 理由② その仕事が「1つしかないもの」を使っている

こちらのほうが見落とされがちです。

2人の担当者が、途中で**同じ1つの資源**を使わなければならない場合、
そこは**一度に1人しか通れません**。たとえば、

- **1台しかない外部ハードウェア**（GPU、アクセラレータ、プリンタ、計測器）
- 1本しかない通信路やファイル
- 排他制御された共有データ

こういう仕事の中身は、たいてい次のように分かれています。

```
[ 準備(CPU) ] → [ 共有資源を使う ] → [ 後始末(CPU) ]
                  ↑ ここだけは順番待ち
```

準備と後始末は2人が同時にできますが、真ん中は必ず順番待ちです。
図にすると、こうなります。

```
準備1    ..11....33....55
共有資源   ..11..22..33..44..55..66     ← 1つしかない。重ならない
準備2    ....22....44....66
```

**それでも2人にする意味はあります。** 1人だけだと、その人が準備や後始末をしている間、
共有資源は遊んでいます。2人いれば、片方が準備をしている裏で、もう片方が資源を使えます。

つまり **2人にする目的は「2倍速くする」ことではなく「共有資源を遊ばせないこと」**。
上限は `1 / 共有資源の使用時間` で頭打ちになります。

次のセルで確かめてください。
「準備30ms（CPU）→ 共有資源60ms（一度に1人）→ 後始末10ms（CPU）」という仕事を、
1人 / 2人 / 4人でやらせます。

In [ ]:
%%writefile ans01b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::mutex shared_hw;      // 1つしかない資源 = 一度に1スレッドしか使えない

void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void worker(int id, int nworker, int njob) {
    for (int f = id; f < njob; f += nworker) {
        wait_ms(30);                                                 // 準備（CPU）
        { std::lock_guard<std::mutex> g(shared_hw); wait_ms(60); }   // 共有資源（順番待ち）
        wait_ms(10);                                                 // 後始末（CPU）
    }
}

void run(int nworker, int njob) {
    auto t0 = steady_clock::now();
    std::vector<std::thread> ts;
    for (int k = 0; k < nworker; k++) ts.emplace_back(worker, k, nworker, njob);
    for (auto& t : ts) t.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();
    std::cout << nworker << "人 : " << ms << " ms  ->  "
              << (1000.0 * njob / ms) << " 件/秒\n";
}

int main() {
    std::cout << "準備30ms + 共有資源60ms(一度に1人) + 後始末10ms、12件\n\n";
    run(1, 12);
    run(2, 12);
    run(4, 12);
    std::cout << "\n共有資源だけの理論下限 = 60ms x 12 = 720 ms -> 16.7 件/秒\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans01b.cpp -o ans01b && ./ans01b

1人 → 2人では大きく速くなり（共有資源の空き時間が埋まる）、
**2人 → 4人ではまったく変わりません**。理論下限に張り付いているからです。

> **`std::mutex`（`lock_guard`）は演習3で扱います。**
> ここでは「1つしかない資源＝一度に1人しか使えない」を表す道具として使っているだけです。
> いまは意味が分からなくても構いません。

まとめると、担当者を増やす前に確かめるべきことは2つです。

1. **その段は本当にボトルネックか**（別の段が詰まっていないか）
2. **その段は本当に並列に動けるか**（コアは足りるか、1つしかない資源を取り合っていないか）

どちらも「増やしてから測る」のではなく、**増やす前に考えられる**ことです。

## 発展課題5 の解答 ―― `join()` を消すと

**プログラムが異常終了します**（`terminate called without an active exception`）。

In [ ]:
%%writefile ans01c.cpp
#include <iostream>
#include <thread>
#include <chrono>

void work() {
    std::this_thread::sleep_for(std::chrono::milliseconds(100));
    std::cout << "スレッドの仕事が終わった\n";
}

int main() {
    std::thread t(work);
    // t.join();          // ← これを消すと？
    std::cout << "main が終わる\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans01c.cpp -o ans01c
!./ans01c || echo "→ 異常終了しました"

`std::thread` のオブジェクトは、**まだ走っているスレッドを抱えたまま破棄されると強制終了する**
決まりになっています。「放置されたスレッドがある」状態を許さない、という設計です。

`join()` は「そのスレッドが終わるまで待って、後片付けをする」処理です。
スレッドを起動したら、必ずどこかで `join()`（または `detach()`）する必要があります。

スレッドを複数まとめて起動したときは、全部について `join()` します。

```cpp
std::vector<std::thread> ts;
...
for (auto& t : ts) t.join();      // 起動した全員を待つ
```

> **`detach()` という選択肢もあります**が、こちらは「もう面倒を見ない」という宣言で、
> そのスレッドが終わったかどうかを知る手段がなくなります。
> 「起動したら `join()`」を基本にしてください。
> スレッドをどう安全に終わらせるかは、演習9で改めて扱います。

---

## 参考：本番のプログラムでは

ハッカソンで読む `yolov3_video_study.cpp` は、まさにこの形をしています。

- スレッドは4本（読み込み1・推論2・表示1）で、キューでつないだ**パイプライン**
- 推論スレッドの中身は「前処理(CPU) → DPU推論 → 後処理(CPU)」
- **DPU はボード上に1個しかない**ので、推論2本は DPU の前で順番待ちになる
  （＝発展課題4の「1つしかない資源」そのもの）
- `main` の冒頭で `hardware_concurrency()` を表示している

いま学んだことが、そのまま「なぜこの構成なのか」の説明になります。
細かい読み方は演習を進めながら扱うので、いまは眺めるだけで構いません。